In [1]:
import transformers
import torch
from datasets import load_dataset
import os 
import sys
from tqdm import tqdm
import json

In [2]:
# from huggingface_hub import login
# login()

## load MMMU dataset & extract keyword from [question, option] (https://huggingface.co/datasets/lmms-lab/MMMU)

In [2]:
server_name = 'ryan'
dataSet = 'clevr' # choose from mmmu, clevr, textocr

# Add the inference directory to the PYTHONPATH
if server_name == 'monet':
    som_path = '/home/monet/meitang/cache-of-thoughts/inference'
    dataDir = '/home/monet/meitang/cache-of-thoughts/data'
    os.environ['HF_HOME'] = '/mnt/data/meitang/.cache/huggingface'
elif server_name == 'ryan':
    som_path = '/home/ryan/meitang/cache-of-thoughts-main/inference'
    dataDir = '/home/ryan/meitang/cache-of-thoughts-main/data'
    os.environ['HF_HOME'] = '/home/ryan/.cache/huggingface'
elif server_name == 'meitang':
    som_path = '/Users/17348/Documents/GitHub/cache-of-thoughts/inference'
    dataDir = '/Users/17348/Documents/GitHub/cache-of-thoughts/data'
    os.environ['HF_HOME'] = '/Users/17348/.cache/huggingface'
else:
    pass # modify accordingly

if som_path not in sys.path:
    sys.path.append(som_path)

os.environ['PYTHONPATH'] = os.environ.get('PYTHONPATH', '') + f":{som_path}"


In [4]:
if dataSet == 'clevr':
    llama_template = """
    Please give me 10 keywords that are present in this context and separate them with commas.
    Make sure you to only return the keywords and say nothing else.
    I have the following contexts:
    - {query}
    """
elif dataSet == 'textocr':
    llama_template = """
    Please give me 10 keywords that are present in this context and separate them with commas.
    Make sure you to only return the keywords and say nothing else.
    Make sure to exclude the word "ANSWER" as keywords
    I have the following contexts:
    - {query}
    """

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import time


model_id = "meta-llama/Meta-Llama-3-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side = "left")
tokenizer.pad_token_id = tokenizer.eos_token_id
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.bfloat16, device_map="auto")
terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]



In [ ]:
output_dir = dataDir.strip('data') + f'/keyword/{dataSet}_gpt/val_keyword'
# create dir if not exists
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

if dataSet == 'textocr':    
    result_path = os.path.join(dataDir, dataSet, f'{dataSet}_val_gpt4o_response_v2.jsonl')
    with open(result_path, 'r') as f:
            gpt_results = f.readlines()

    llama_keywords = []
    for i, line in enumerate(tqdm(gpt_results)):
        message = [
                {"role": "system", "content": "You are a helpful chatbot that assists users in generating keywords from conversations. You have been given a conversation and need to generate keywords from it."},
                {"role": "user", "content": llama_template.format(query=line) },
            ]
        texts = tokenizer.apply_chat_template(message, add_generation_prompt=True, tokenize=False)
        inputs = tokenizer(texts, padding="longest", return_tensors="pt").to(model.device)
        # inputs = {key: val. for key, val in inputs.items()}
        temp_texts=tokenizer.batch_decode(inputs["input_ids"], skip_special_tokens=True)


        start_time = time.time()
        gen_tokens = model.generate(
            **inputs, 
            max_new_tokens=128, 
            pad_token_id=tokenizer.eos_token_id, 
            eos_token_id=terminators,
            do_sample=True,
            temperature=0.6,
            top_p=0.9
        )

        gen_text = tokenizer.batch_decode(gen_tokens, skip_special_tokens=True)
        gen_text = [i[len(temp_texts[idx]):] for idx, i in enumerate(gen_text)]

        llama_keywords.extend(gen_text)

        # write to file every 10 batches or at the end
        if i % 10 == 0 or i == len(gpt_results) - 1:
            with open(os.path.join(output_dir, f"llama_keywords_{i}.txt"), "w") as f:
                f.write("\n".join(llama_keywords))
            llama_keywords = []
elif dataSet == 'clevr':
    support_file = os.path.join(dataDir, dataSet, 'support.json')
    with open(support_file, 'r') as f:
            support_meta = json.load(f)
    validation_dataset = support_meta
    llama_keywords = []
    for i, each_query in enumerate(tqdm(validation_dataset)):
        property_name, exact_name  = each_query['question'].split(': ')
        prompt = f'How many objects in the image have the {exact_name} {property_name}'
        message = [
                {"role": "system", "content": "You are a helpful chatbot that assists users in generating keywords from conversations. You have been given a conversation and need to generate keywords from it."},
                {"role": "user", "content": llama_template.format(query=prompt) },
            ]
        texts = tokenizer.apply_chat_template(message, add_generation_prompt=True, tokenize=False)
        inputs = tokenizer(texts, padding="longest", return_tensors="pt").to(model.device)
        # inputs = {key: val. for key, val in inputs.items()}
        temp_texts=tokenizer.batch_decode(inputs["input_ids"], skip_special_tokens=True)


        start_time = time.time()
        gen_tokens = model.generate(
            **inputs, 
            max_new_tokens=128, 
            pad_token_id=tokenizer.eos_token_id, 
            eos_token_id=terminators,
            do_sample=True,
            temperature=0.6,
            top_p=0.9
        )

        gen_text = tokenizer.batch_decode(gen_tokens, skip_special_tokens=True)
        gen_text = [i[len(temp_texts[idx]):] for idx, i in enumerate(gen_text)]

        llama_keywords.extend(gen_text)

        # write to file every 10 batches or at the end
        if i % 10 == 0 or i == len(validation_dataset) - 1:
            with open(os.path.join(output_dir, f"llama_keywords_{i}.txt"), "w") as f:
                f.write("\n".join(llama_keywords))
            llama_keywords = []

In [ ]:
torch.cuda.empty_cache()

In [ ]:
print(torch.cuda.memory_summary())

# Check total allocated memory on the current device
print(f"Allocated memory: {torch.cuda.memory_allocated() / 1024**3} GB")

# Check total reserved memory on the current device
print(f"Reserved memory: {torch.cuda.memory_reserved() / 1024**3} GB")